In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

In [2]:
views = pd.read_csv("./data/views.csv")
views

,user_id,exp_group,recommendations,timestamp
0,128381,control,[3644 4529 4704 5294 4808],1654030803
1,146885,test,[1399 1076 797 7015 5942],1654030811
2,50948,test,[2315 3037 1861 6567 4093],1654030825
3,37703,test,[2842 1949 162 1588 6794],1654030826
4,14661,test,[2395 5881 5648 3417 673],1654030829
...,...,...,...,...
193290,158267,test,[1733 6834 4380 1915 1627],1655240340
193291,63527,control,[2454 191 3873 6404 1588],1655240347
193292,52169,test,[1368 1709 1616 798 5305],1655240354
193293,142402,test,[5895 6984 1978 6548 6106],1655240373


In [3]:
likes = pd.read_csv("./data/likes.csv")
likes

,user_id,post_id,timestamp
0,128381,4704,1654030804
1,146885,1399,1654030816
2,50948,2315,1654030828
3,14661,673,1654030831
4,37703,1588,1654030833
...,...,...,...
230171,31851,5964,1655243535
230172,51512,1498,1655243537
230173,34017,5009,1655243573
230174,13267,1787,1655243692


Начнём с проверки разбиения групп.

Во-первых, у нас нет таблички соответствия пользователь-группа, так как мы на самом деле определяли группу пользователя прямо перед подготовкой рекомендаций.

Может показаться, что это гарантирует нам однозначное соответствие групп для пользователей, но давайте это проверим. В реальности бывают лаги системы (например, если мы запрашиваем группу пользователя, то не всегда можем получить ответ) и это свойство не выполняется. Проверьте, нет ли у нас пользователей, которые попали в обе группы. Если их совсем немного, удалите их из обеих выборок (если бы было много, то надо было разбираться, что пошло не так).

Теперь, когда мы знаем про однозначное соответствие, сделайте табличку пользователь-группа. Проверьте, что группы получаются одинакового размера по пользователям. Для этого можно посчитать долю каждой группы, а ещё можно применить критерий для долей (биномиальный тест), чтобы проверить соответствие этой доли ожидаемым 0.5.

In [9]:
user_exp_group_count = views.groupby("user_id")["exp_group"].nunique()
user_exp_group_count

user_id
200       1
201       1
202       1
212       1
213       1
         ..
168538    1
168541    1
168544    1
168545    1
168552    1
Name: exp_group, Length: 65013, dtype: int64

In [19]:
duplicates_idx = user_exp_group_count[user_exp_group_count > 1].index
duplicates = views[views["user_id"].isin(duplicates_idx)]
print("duplicates shape: ", duplicates.shape)
duplicates.head()


duplicates shape:  (27, 4)


,user_id,exp_group,recommendations,timestamp
1311,148670,test,[2992 1368 1261 3901 4471],1654039282
6179,142283,control,[1109 101 5288 4941 132],1654069529
29724,148670,test,[5053 1563 7194 633 1392],1654217190
30748,148670,test,[7128 1023 1388 6807 5945],1654223589
39653,55788,test,[3747 6638 5214 2801 5740],1654279384


In [20]:
non_duplicates_idx = user_exp_group_count[user_exp_group_count == 1].index
filtered_views = views[views["user_id"].isin(non_duplicates_idx)]
print("filtered_views shape: ", filtered_views.shape)
filtered_views

filtered_views shape:  (193268, 4)


,user_id,exp_group,recommendations,timestamp
0,128381,control,[3644 4529 4704 5294 4808],1654030803
1,146885,test,[1399 1076 797 7015 5942],1654030811
2,50948,test,[2315 3037 1861 6567 4093],1654030825
3,37703,test,[2842 1949 162 1588 6794],1654030826
4,14661,test,[2395 5881 5648 3417 673],1654030829
...,...,...,...,...
193290,158267,test,[1733 6834 4380 1915 1627],1655240340
193291,63527,control,[2454 191 3873 6404 1588],1655240347
193292,52169,test,[1368 1709 1616 798 5305],1655240354
193293,142402,test,[5895 6984 1978 6548 6106],1655240373


In [16]:
print(f"Users in CONTROL group: {filtered_views[filtered_views['exp_group'] == 'control']['user_id'].nunique()}")
print(f"Users in TEST group: {filtered_views[filtered_views['exp_group'] == 'test']['user_id'].nunique()}")

Users in CONTROL group: 32350
Users in TEST group: 32659


**Есть немного пользователей сразу в 2 группах, группы равного размера.**

---

Теперь давайте попробуем оценить, а улучшаются ли наши метрики в тестовой группе.

Наши данные о показах и кликах хранятся в разных табличках и просто так их не получается сджойнить. Давайте оставим это на потом, а пока попробуем оценить более простыми метриками. Подумайте, какие метрики можно было бы посчитать без джойна показов и кликов.

Соберите табличку, в которой будут пользователи, попавшие в наш эксперимент (то есть те, для которых мы строили хотя бы одну рекомендацию). Посчитайте, кто из них сколько лайков сделал. Наверняка будут пользователи, которые не сделали ни один лайк.

Посчитайте долю пользователей, которая сделала хотя бы один лайк за время эксперимента без разбивки на группы.



In [23]:
filtered_views.describe()

,user_id,timestamp
count,193268.000000,1.932680e+05
mean,85295.466089,1.654635e+09
std,48874.216543,3.487127e+05
min,200.000000,1.654031e+09
25%,41737.000000,1.654334e+09
50%,85944.000000,1.654635e+09
75%,127873.250000,1.654936e+09
max,168552.000000,1.655240e+09


In [24]:
filtered_views.isna().sum()

user_id            0
exp_group          0
recommendations    0
timestamp          0
dtype: int64

In [ ]:
exp_users_idx = filtered_views["user_id"].unique()
exp_users_idx

array([128381, 146885,  50948, ...,   3615, 119630, 142402], dtype=int64)

In [33]:
exp_users_likes = likes[likes["user_id"].isin(exp_users_idx)]['user_id'].unique()

print(f"Доля пользователей поставиших хотя бы 1 лайк: {len(exp_users_likes) * 100 /len(exp_users_idx):.1f}%")

Доля пользователей поставиших хотя бы 1 лайк: 89.5%


---

А теперь давайте оценим, различаются ли число лайков между группами.

Будем проверять две метрики:

**Доля пользователей с хотя бы одним лайком по группам**

In [37]:
control_users_idx = filtered_views[filtered_views["exp_group"] == 'control']["user_id"].unique()
control_users_likes_idx = likes[likes["user_id"].isin(control_users_idx)]['user_id'].unique()
control_likes_share = len(control_users_likes_idx) * 100 /len(control_users_idx)
print(f"Доля пользователей поставиших хотя бы 1 лайк в контрольной группе: {control_likes_share:.2f}%")

Доля пользователей поставиших хотя бы 1 лайк в контрольной группе: 89.13%


In [39]:
test_users_idx = filtered_views[filtered_views["exp_group"] == 'test']["user_id"].unique()
test_users_likes_idx = likes[likes["user_id"].isin(test_users_idx)]['user_id'].unique()
test_likes_share = len(test_users_likes_idx) * 100 /len(test_users_idx)
print(f"Доля пользователей поставиших хотя бы 1 лайк в тестовой группе: {test_likes_share:.2f}%")

Доля пользователей поставиших хотя бы 1 лайк в тестовой группе: 89.82%


Применим Z-критерий для долей

In [40]:
control_users_total = len(control_users_idx)
test_users_total = len(test_users_idx)

control_users_likes_total = len(control_users_likes_idx)
test_users_likes_total = len(test_users_likes_idx)

In [45]:
from statsmodels.stats.proportion import proportions_ztest
_, p_value = proportions_ztest(
    [control_users_likes_total, test_users_likes_total],
    [control_users_total, test_users_total],
    alternative = "two-sided"
)

print("P-value: ", p_value)
if p_value < 0.05:
    print("Отвергаем нулевую гипотезу. Имеется статистически значимое различие в долях между группами")
else:
    print("Не отвергаем нулевую гипотезу. Статистически значимого различия в долях между группами нет")

P-value:  0.00445475668548642
Отвергаем нулевую гипотезу. Имеется статистически значимое различие в долях между группами


**Число лайков на пользователя**

In [53]:
control_users_likes_count = likes[likes["user_id"].isin(control_users_idx)].groupby("user_id").size().values
test_users_likes_count = likes[likes["user_id"].isin(test_users_idx)].groupby("user_id").size().values

In [52]:
control_users_likes_count

array([4, 5, 3, ..., 1, 2, 2], dtype=int64)

In [54]:
test_users_likes_count

array([1, 3, 2, ..., 4, 5, 5], dtype=int64)

Проверим данные на нормальность

In [ ]:
### Проверим распределение лайков в контрольной группе

_, p_value = stats.shapiro(control_users_likes_count)

if p_value < 0.05:
    print("Распределение лайков в контрольной группе не является нормальным")
else:
    print("Распределение лайков в контрольной группе является нормальным")

Распределение лайков в контрольной группе не является нормальным


c:\Users\fedor\KC_Final_RecSys\venv\lib\site-packages\scipy\stats\_morestats.py:1816: UserWarning: p-value may not be accurate for N > 5000.
  warnings.warn("p-value may not be accurate for N > 5000.")


In [61]:
### Проверим распределение лайков в тестовой группе

_, p_value = stats.shapiro(test_users_likes_count)

if p_value < 0.05:
    print("Распределение лайков в тестовой группе не является нормальным")
else:
    print("Распределение лайков в тестовой группе является нормальным")

Распределение лайков в тестовой группе не является нормальным


c:\Users\fedor\KC_Final_RecSys\venv\lib\site-packages\scipy\stats\_morestats.py:1816: UserWarning: p-value may not be accurate for N > 5000.
  warnings.warn("p-value may not be accurate for N > 5000.")


Так как распределение не является нормальным, мы не можем использовать t критерий.

Используем критерий Манна-Уитни

In [62]:
_, p_value = stats.mannwhitneyu(
    control_users_likes_count,
    test_users_likes_count
)

print("P-value: ", p_value)

if p_value < 0.05:
    print("Отвергаем нулевую гипотезу. Имеется статистически значимое различие в среднем количестве лайков на пользователя между группами")
else:
    print("Не отвергаем нулевую гипотезу. Статистически значимого различия в среднем количестве лайков на пользователя между группами нет")

P-value:  0.0016817776764960254
Отвергаем нулевую гипотезу. Имеется статистически значимое различие в среднем количестве лайков на пользователя между группами


----

Посчитаем hitrate - долю рекомендаций, которые пользователь лайкнул

In [66]:
views

,user_id,exp_group,recommendations,timestamp
0,128381,control,[3644 4529 4704 5294 4808],1654030803
1,146885,test,[1399 1076 797 7015 5942],1654030811
2,50948,test,[2315 3037 1861 6567 4093],1654030825
3,37703,test,[2842 1949 162 1588 6794],1654030826
4,14661,test,[2395 5881 5648 3417 673],1654030829
...,...,...,...,...
193290,158267,test,[1733 6834 4380 1915 1627],1655240340
193291,63527,control,[2454 191 3873 6404 1588],1655240347
193292,52169,test,[1368 1709 1616 798 5305],1655240354
193293,142402,test,[5895 6984 1978 6548 6106],1655240373


In [67]:
views["recommendations"] = views["recommendations"].apply(lambda x: set(map(int, x.strip("[]").split())))

In [68]:
views

,user_id,exp_group,recommendations,timestamp
0,128381,control,"{4704, 4808, 5294, 4529, 3644}",1654030803
1,146885,test,"{7015, 1076, 5942, 1399, 797}",1654030811
2,50948,test,"{1861, 6567, 2315, 3037, 4093}",1654030825
3,37703,test,"{162, 6794, 1588, 2842, 1949}",1654030826
4,14661,test,"{673, 3417, 5648, 5881, 2395}",1654030829
...,...,...,...,...
193290,158267,test,"{1915, 1733, 6834, 1627, 4380}",1655240340
193291,63527,control,"{3873, 6404, 1588, 2454, 191}",1655240347
193292,52169,test,"{1709, 1616, 1368, 5305, 798}",1655240354
193293,142402,test,"{1978, 5895, 6984, 6548, 6106}",1655240373


In [76]:
merged = views.merge(likes, on = "user_id", how = "left", suffixes = ("_views", "_likes"))
merged

,user_id,exp_group,recommendations,timestamp_views,post_id,timestamp_likes
0,128381,control,"{4704, 4808, 5294, 4529, 3644}",1654030803,4704.0,1.654031e+09
1,128381,control,"{4704, 4808, 5294, 4529, 3644}",1654030803,5294.0,1.654031e+09
2,128381,control,"{4704, 4808, 5294, 4529, 3644}",1654030803,3608.0,1.655049e+09
3,128381,control,"{4704, 4808, 5294, 4529, 3644}",1654030803,2542.0,1.655049e+09
4,128381,control,"{4704, 4808, 5294, 4529, 3644}",1654030803,4165.0,1.655053e+09
...,...,...,...,...,...,...
1017168,52169,test,"{1709, 1616, 1368, 5305, 798}",1655240354,1709.0,1.655240e+09
1017169,142402,test,"{1978, 5895, 6984, 6548, 6106}",1655240373,6548.0,1.655240e+09
1017170,72259,control,"{3587, 3811, 6117, 6567, 1255}",1655240388,1712.0,1.654305e+09
1017171,72259,control,"{3587, 3811, 6117, 6567, 1255}",1655240388,5070.0,1.654309e+09


Посмотрим на пропуски

In [77]:
merged.isna().sum()

user_id                0
exp_group              0
recommendations        0
timestamp_views        0
post_id            10531
timestamp_likes    10531
dtype: int64

In [78]:
merged.dropna(inplace = True)

In [79]:
merged["timestamp_views"] = pd.to_datetime(merged["timestamp_views"], unit = "s")
merged["timestamp_likes"] = pd.to_datetime(merged["timestamp_likes"], unit = "s")
merged

,user_id,exp_group,recommendations,timestamp_views,post_id,timestamp_likes
0,128381,control,"{4704, 4808, 5294, 4529, 3644}",2022-05-31 21:00:03,4704.0,2022-05-31 21:00:04
1,128381,control,"{4704, 4808, 5294, 4529, 3644}",2022-05-31 21:00:03,5294.0,2022-05-31 21:00:38
2,128381,control,"{4704, 4808, 5294, 4529, 3644}",2022-05-31 21:00:03,3608.0,2022-06-12 15:55:27
3,128381,control,"{4704, 4808, 5294, 4529, 3644}",2022-05-31 21:00:03,2542.0,2022-06-12 15:55:42
4,128381,control,"{4704, 4808, 5294, 4529, 3644}",2022-05-31 21:00:03,4165.0,2022-06-12 16:53:26
...,...,...,...,...,...,...
1017168,52169,test,"{1709, 1616, 1368, 5305, 798}",2022-06-14 20:59:14,1709.0,2022-06-14 20:59:29
1017169,142402,test,"{1978, 5895, 6984, 6548, 6106}",2022-06-14 20:59:33,6548.0,2022-06-14 20:59:34
1017170,72259,control,"{3587, 3811, 6117, 6567, 1255}",2022-06-14 20:59:48,1712.0,2022-06-04 01:11:51
1017171,72259,control,"{3587, 3811, 6117, 6567, 1255}",2022-06-14 20:59:48,5070.0,2022-06-04 02:09:50


Оставим только записи, там где лайкнутый пост находится в рекомендациях

In [80]:
merged["liked_in_recommendations"] = merged.apply(lambda row: row["post_id"] in row["recommendations"], axis = 1)
merged

,user_id,exp_group,recommendations,timestamp_views,post_id,timestamp_likes,liked_in_recommendations
0,128381,control,"{4704, 4808, 5294, 4529, 3644}",2022-05-31 21:00:03,4704.0,2022-05-31 21:00:04,True
1,128381,control,"{4704, 4808, 5294, 4529, 3644}",2022-05-31 21:00:03,5294.0,2022-05-31 21:00:38,True
2,128381,control,"{4704, 4808, 5294, 4529, 3644}",2022-05-31 21:00:03,3608.0,2022-06-12 15:55:27,False
3,128381,control,"{4704, 4808, 5294, 4529, 3644}",2022-05-31 21:00:03,2542.0,2022-06-12 15:55:42,False
4,128381,control,"{4704, 4808, 5294, 4529, 3644}",2022-05-31 21:00:03,4165.0,2022-06-12 16:53:26,False
...,...,...,...,...,...,...,...
1017168,52169,test,"{1709, 1616, 1368, 5305, 798}",2022-06-14 20:59:14,1709.0,2022-06-14 20:59:29,True
1017169,142402,test,"{1978, 5895, 6984, 6548, 6106}",2022-06-14 20:59:33,6548.0,2022-06-14 20:59:34,True
1017170,72259,control,"{3587, 3811, 6117, 6567, 1255}",2022-06-14 20:59:48,1712.0,2022-06-04 01:11:51,False
1017171,72259,control,"{3587, 3811, 6117, 6567, 1255}",2022-06-14 20:59:48,5070.0,2022-06-04 02:09:50,False


In [81]:
merged = merged[merged["liked_in_recommendations"] == True]
merged


,user_id,exp_group,recommendations,timestamp_views,post_id,timestamp_likes,liked_in_recommendations
0,128381,control,"{4704, 4808, 5294, 4529, 3644}",2022-05-31 21:00:03,4704.0,2022-05-31 21:00:04,True
1,128381,control,"{4704, 4808, 5294, 4529, 3644}",2022-05-31 21:00:03,5294.0,2022-05-31 21:00:38,True
7,146885,test,"{7015, 1076, 5942, 1399, 797}",2022-05-31 21:00:11,1399.0,2022-05-31 21:00:16,True
11,50948,test,"{1861, 6567, 2315, 3037, 4093}",2022-05-31 21:00:25,2315.0,2022-05-31 21:00:28,True
16,37703,test,"{162, 6794, 1588, 2842, 1949}",2022-05-31 21:00:26,1588.0,2022-05-31 21:00:33,True
...,...,...,...,...,...,...,...
1017160,119630,test,"{7077, 3143, 1577, 4588, 599}",2022-06-14 20:58:57,599.0,2022-06-14 20:59:27,True
1017163,158267,test,"{1915, 1733, 6834, 1627, 4380}",2022-06-14 20:59:00,6834.0,2022-06-14 20:59:01,True
1017167,63527,control,"{3873, 6404, 1588, 2454, 191}",2022-06-14 20:59:07,3873.0,2022-06-14 20:59:18,True
1017168,52169,test,"{1709, 1616, 1368, 5305, 798}",2022-06-14 20:59:14,1709.0,2022-06-14 20:59:29,True


Оставим только те записи, где пост лайкнули после получения рекомендаций, но не позже, чем через час после этого.

In [84]:
filtered_total_data = merged[
    (merged["timestamp_views"] <= merged["timestamp_likes"]) &
    (merged["timestamp_likes"] <= merged["timestamp_views"] + pd.Timedelta(hours = 1))
]

filtered_total_data

,user_id,exp_group,recommendations,timestamp_views,post_id,timestamp_likes,liked_in_recommendations
0,128381,control,"{4704, 4808, 5294, 4529, 3644}",2022-05-31 21:00:03,4704.0,2022-05-31 21:00:04,True
1,128381,control,"{4704, 4808, 5294, 4529, 3644}",2022-05-31 21:00:03,5294.0,2022-05-31 21:00:38,True
7,146885,test,"{7015, 1076, 5942, 1399, 797}",2022-05-31 21:00:11,1399.0,2022-05-31 21:00:16,True
11,50948,test,"{1861, 6567, 2315, 3037, 4093}",2022-05-31 21:00:25,2315.0,2022-05-31 21:00:28,True
16,37703,test,"{162, 6794, 1588, 2842, 1949}",2022-05-31 21:00:26,1588.0,2022-05-31 21:00:33,True
...,...,...,...,...,...,...,...
1017160,119630,test,"{7077, 3143, 1577, 4588, 599}",2022-06-14 20:58:57,599.0,2022-06-14 20:59:27,True
1017163,158267,test,"{1915, 1733, 6834, 1627, 4380}",2022-06-14 20:59:00,6834.0,2022-06-14 20:59:01,True
1017167,63527,control,"{3873, 6404, 1588, 2454, 191}",2022-06-14 20:59:07,3873.0,2022-06-14 20:59:18,True
1017168,52169,test,"{1709, 1616, 1368, 5305, 798}",2022-06-14 20:59:14,1709.0,2022-06-14 20:59:29,True


In [86]:
filtered_total_data = filtered_total_data.sort_values(by = ["timestamp_views"], ascending= False)
filtered_total_data

,user_id,exp_group,recommendations,timestamp_views,post_id,timestamp_likes,liked_in_recommendations
1017169,142402,test,"{1978, 5895, 6984, 6548, 6106}",2022-06-14 20:59:33,6548.0,2022-06-14 20:59:34,True
1017168,52169,test,"{1709, 1616, 1368, 5305, 798}",2022-06-14 20:59:14,1709.0,2022-06-14 20:59:29,True
1017167,63527,control,"{3873, 6404, 1588, 2454, 191}",2022-06-14 20:59:07,3873.0,2022-06-14 20:59:18,True
1017163,158267,test,"{1915, 1733, 6834, 1627, 4380}",2022-06-14 20:59:00,6834.0,2022-06-14 20:59:01,True
1017160,119630,test,"{7077, 3143, 1577, 4588, 599}",2022-06-14 20:58:57,599.0,2022-06-14 20:59:27,True
...,...,...,...,...,...,...,...
16,37703,test,"{162, 6794, 1588, 2842, 1949}",2022-05-31 21:00:26,1588.0,2022-05-31 21:00:33,True
11,50948,test,"{1861, 6567, 2315, 3037, 4093}",2022-05-31 21:00:25,2315.0,2022-05-31 21:00:28,True
7,146885,test,"{7015, 1076, 5942, 1399, 797}",2022-05-31 21:00:11,1399.0,2022-05-31 21:00:16,True
1,128381,control,"{4704, 4808, 5294, 4529, 3644}",2022-05-31 21:00:03,5294.0,2022-05-31 21:00:38,True


In [88]:
hitrate_df = filtered_total_data.groupby(["user_id", "timestamp_views"]).size().reset_index(name = "like_count")
hitrate_df

,user_id,timestamp_views,like_count
0,200,2022-06-12 04:44:07,1
1,201,2022-06-05 15:58:24,1
2,201,2022-06-08 04:53:59,2
3,202,2022-06-06 01:55:54,2
4,212,2022-06-01 16:59:15,3
...,...,...,...
137866,168541,2022-06-12 06:43:57,2
137867,168545,2022-06-01 02:34:05,2
137868,168545,2022-06-03 22:28:01,1
137869,168545,2022-06-05 06:56:24,1


In [90]:
views = views.rename(columns = {"timestamp": "timestamp_views"})
views["timestamp_views"] = pd.to_datetime(views["timestamp_views"], unit = "s")
views

,user_id,exp_group,recommendations,timestamp_views
0,128381,control,"{4704, 4808, 5294, 4529, 3644}",2022-05-31 21:00:03
1,146885,test,"{7015, 1076, 5942, 1399, 797}",2022-05-31 21:00:11
2,50948,test,"{1861, 6567, 2315, 3037, 4093}",2022-05-31 21:00:25
3,37703,test,"{162, 6794, 1588, 2842, 1949}",2022-05-31 21:00:26
4,14661,test,"{673, 3417, 5648, 5881, 2395}",2022-05-31 21:00:29
...,...,...,...,...
193290,158267,test,"{1915, 1733, 6834, 1627, 4380}",2022-06-14 20:59:00
193291,63527,control,"{3873, 6404, 1588, 2454, 191}",2022-06-14 20:59:07
193292,52169,test,"{1709, 1616, 1368, 5305, 798}",2022-06-14 20:59:14
193293,142402,test,"{1978, 5895, 6984, 6548, 6106}",2022-06-14 20:59:33


In [91]:
final_data = views.merge(hitrate_df, on = ["user_id", "timestamp_views"], how = "left")
final_data

,user_id,exp_group,recommendations,timestamp_views,like_count
0,128381,control,"{4704, 4808, 5294, 4529, 3644}",2022-05-31 21:00:03,2.0
1,146885,test,"{7015, 1076, 5942, 1399, 797}",2022-05-31 21:00:11,1.0
2,50948,test,"{1861, 6567, 2315, 3037, 4093}",2022-05-31 21:00:25,1.0
3,37703,test,"{162, 6794, 1588, 2842, 1949}",2022-05-31 21:00:26,2.0
4,14661,test,"{673, 3417, 5648, 5881, 2395}",2022-05-31 21:00:29,1.0
...,...,...,...,...,...
193290,158267,test,"{1915, 1733, 6834, 1627, 4380}",2022-06-14 20:59:00,1.0
193291,63527,control,"{3873, 6404, 1588, 2454, 191}",2022-06-14 20:59:07,1.0
193292,52169,test,"{1709, 1616, 1368, 5305, 798}",2022-06-14 20:59:14,1.0
193293,142402,test,"{1978, 5895, 6984, 6548, 6106}",2022-06-14 20:59:33,1.0


In [92]:
final_data.isna().sum()

user_id                0
exp_group              0
recommendations        0
timestamp_views        0
like_count         55424
dtype: int64

In [93]:
final_data["like_count"] = final_data["like_count"].fillna(0).astype(int)

In [94]:
final_data.isna().sum()

user_id            0
exp_group          0
recommendations    0
timestamp_views    0
like_count         0
dtype: int64

In [96]:
hitrate = (final_data["like_count"] > 0).mean()

print(f"Hitrate: {hitrate:.4f}")


Hitrate: 0.7133


---

А теперь давайте оценим различие между группами и значимость. z-критерий для долей мы здесь не можем применять, так как у нас в каждой выборке один и тот же пользователь может встречаться несколько раз. Давайте применим бакетный подход (то есть перейдём к бакетам и по ним оценим значимость), чтобы посчитать групповой hitrate (или CTR) — доля hitrate по группе/бакету. Используйте 100 бакетов. Уровень значимости останется тем же на уровне 0.05.

In [98]:
final_data["bucket"] = np.random.randint(0, 100, size=len(final_data))

final_data["clicked"] = final_data["like_count"] > 0

In [99]:
control_data = final_data[final_data["exp_group"] == 'control']
test_data = final_data[final_data["exp_group"] == 'test']

In [100]:
control_hitrate_by_bucket = control_data.groupby('bucket')["clicked"].mean()
test_hitrate_by_bucket = test_data.groupby('bucket')["clicked"].mean()

In [103]:
hitrate_difference = test_hitrate_by_bucket.mean() - control_hitrate_by_bucket.mean()
hitrate_difference_pct = hitrate_difference * 100

In [104]:
_, p_value = stats.mannwhitneyu(control_hitrate_by_bucket, test_hitrate_by_bucket)

print("p-value:", p_value)

if p_value < 0.05:
    if hitrate_difference_pct > 0:
        print(f"В тестовой группе hitrate выше на {int(hitrate_difference_pct)} п.п. (после бакетирования), есть статистически значимая разница.")
    else:
        print(f"В тестовой группе hitrate ниже на {abs(int(hitrate_difference_pct))} п.п. (после бакетирования), есть статистически значимая разница.")
else:
    print(f"Нет статистически значимой разницы между hitrate в группах.")

p-value: 1.545344967162861e-08
В тестовой группе hitrate выше на 1 п.п. (после бакетирования), есть статистически значимая разница.
